# Dataset Exploration

**Purpose:** Explore and understand the Kaggle ASL Alphabet dataset before preprocessing.

In [ ]:
# Cell 1 — Imports and setup
import os
import sys
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.abspath(".."))
import config

print("Libraries loaded")
print("Dataset path:", config.KAGGLE_ASL_PATH)
print("Path exists:", os.path.exists(config.KAGGLE_ASL_PATH))

In [ ]:
# Cell 2 — Check folder structure
print("Checking dataset folder structure...")
print("-" * 50)

classes = sorted([c for c in os.listdir(config.KAGGLE_ASL_PATH)
                  if os.path.isdir(os.path.join(config.KAGGLE_ASL_PATH, c))])

print(f"Total classes found: {len(classes)}")
print(f"Classes: {classes}")

In [ ]:
# Cell 3 — Count images per class
print("Counting images per class...")
print("-" * 50)

class_counts = {}
total = 0

for cls in classes:
    cls_path = os.path.join(config.KAGGLE_ASL_PATH, cls)
    count = len([f for f in os.listdir(cls_path)
                 if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    class_counts[cls] = count
    total += count
    print(f"  {cls:10s}: {count:5d} images")

print("-" * 50)
print(f"Total images:      {total:,}")
print(f"Total classes:     {len(class_counts)}")
print(f"Average per class: {total // len(class_counts):,}")
print(f"Min: {min(class_counts.values())} — {min(class_counts, key=class_counts.get)}")
print(f"Max: {max(class_counts.values())} — {max(class_counts, key=class_counts.get)}")

In [ ]:
# Cell 4 — Check image properties
print("Checking image properties...")
print("-" * 50)

first_class = classes[0]
first_class_path = os.path.join(config.KAGGLE_ASL_PATH, first_class)
sample_images = os.listdir(first_class_path)[:3]

for img_name in sample_images:
    img_path = os.path.join(first_class_path, img_name)
    img = cv2.imread(img_path)
    if img is not None:
        print(f"  {img_name}: shape={img.shape}, dtype={img.dtype}")

print("-" * 50)
print("Image format: BGR — will convert to RGB for MediaPipe")
print("Ready for preprocessing")

In [ ]:
# Cell 5 — Class balance check
print("Class balance analysis...")
print("-" * 50)

counts = list(class_counts.values())
mean = np.mean(counts)
std = np.std(counts)

print(f"Mean samples per class: {mean:.0f}")
print(f"Std deviation:          {std:.0f}")
print(f"Min samples:            {min(counts)} — {min(class_counts, key=class_counts.get)}")
print(f"Max samples:            {max(counts)} — {max(class_counts, key=class_counts.get)}")
print(f"Imbalance ratio:        {max(counts)/min(counts):.2f}x")

if max(counts) / min(counts) > 2:
    print("\n⚠️  Dataset is imbalanced — augmentation needed in Step 4")
else:
    print("\n✅  Dataset is reasonably balanced")

In [ ]:
# Cell 6 — Plot class distribution
plt.figure(figsize=(16, 6))
colors = ['#534AB7' if v == max(class_counts.values())
          else '#C00000' if v == min(class_counts.values())
          else '#1F4E79' for v in class_counts.values()]

plt.bar(class_counts.keys(), class_counts.values(), color=colors, alpha=0.85)
plt.title('Signify — ASL Alphabet Dataset Class Distribution',
          fontsize=14, fontweight='bold')
plt.xlabel('ASL Class', fontsize=12)
plt.ylabel('Number of Images', fontsize=12)
plt.xticks(rotation=45)
plt.axhline(y=np.mean(list(class_counts.values())),
            color='orange', linestyle='--', alpha=0.7, label='Mean')
plt.legend()
plt.tight_layout()

chart_path = os.path.join(config.BASE_DIR, "notebooks", "dataset_distribution.png")
plt.savefig(chart_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Chart saved to: {chart_path}")

In [ ]:
# Cell 7 — Display sample images from all 29 classes
fig, axes = plt.subplots(3, 10, figsize=(20, 7))
axes = axes.flatten()

for i, cls in enumerate(classes):
    cls_path = os.path.join(config.KAGGLE_ASL_PATH, cls)
    img_file = os.listdir(cls_path)[0]
    img = cv2.imread(os.path.join(cls_path, img_file))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    axes[i].imshow(img_rgb)
    axes[i].set_title(cls, fontsize=10, fontweight='bold')
    axes[i].axis('off')

for i in range(len(classes), len(axes)):
    axes[i].axis('off')

plt.suptitle('Signify — Sample Images from All 29 ASL Classes',
             fontsize=13, fontweight='bold')
plt.tight_layout()

samples_path = os.path.join(config.BASE_DIR, "notebooks", "sample_images.png")
plt.savefig(samples_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Sample images saved to: {samples_path}")

In [ ]:
# Cell 8 — Final summary
print("=" * 60)
print("DATASET EXPLORATION SUMMARY")
print("=" * 60)
print(f"Dataset:       Kaggle ASL Alphabet")
print(f"Total images:  {total:,}")
print(f"Total classes: {len(classes)}")
print(f"Image format:  JPG, 200x200 pixels, BGR")
print(f"Class range:   {min(counts)} to {max(counts)} images")
print(f"Imbalance:     nothing class needs augmentation")
print(f"")
print(f"✅ Dataset verified and ready for Step 3 — Preprocessing")
print("=" * 60)